# Trabajo Práctico  - Calibración y Pose de la Cámara

## Resolución

## GRUPO: NOMBRE DEL GRUPO

# Integrantes
* integrante
* integrante

### Ejercicio 1 - DLT ecuaciones formales

Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

Encontrar el vector $\mathbf{m}$ que resuelve la minimización de la ecuación 6.

### Ejercicio 2 - Calibración con puntos 3D

a) Utilizando los datos de la carpeta $\texttt{calib/3d}$ calcular $\mathbf{R}$,  $\mathbf{t}$ y $\mathbf{K}$.

Calcular un nuevo set de parámetros para 

b) $\alpha_u = \alpha_v$ 

c) $K_{12}=0$



In [6]:
import numpy as np
import os
import alc

carpeta = os.path.join("calib", "3d")
coord_pix_fotos = np.loadtxt(os.path.join(carpeta, "detected_corners.txt"))
coord_reales_puntos = np.loadtxt(os.path.join(carpeta, "p_W_corners.txt"), delimiter=",")

# 1 Q = 1 foto
# u1,v1,u2,v2... 
#a)
def obtenerQ3d(imgPts,objPts):
    #imgPts es un array que representa 1 sola foto
    #objPts es la matriz de coordenadas U,V,W de varios puntos en el mundo real
    n = len(objPts)
    Q = np.zeros((2*n,12))
    for i in range(n):
        #fila 1 del punto (coordenadas u)
        Q[2*i,:3] = objPts[i]
        Q[2*i,3] = 1
        u_i,v_i = imgPts[2*i], imgPts[2*i+1]
        Q[2*i,8:11] = -u_i*objPts[i]
        Q[2*i,11] = -u_i
        #fila 2 del mismo punto (coordenadas v)
        Q[2*i+1,4:7] = objPts[i]
        Q[2*i+1,7] = 1
        Q[2*i+1, 8:11] = -v_i*objPts[i]
        Q[2*i+1,11] = -v_i
    return Q
Q = obtenerQ3d(coord_pix_fotos[0],coord_reales_puntos)

In [28]:
from alc import diagRH, multiplicarMatrices
#b)
def resolverDLT3D(Q):
    S,D = diagRH(multiplicarMatrices(Q.T,Q))
    m = S[:,11] # 12x1, M 3x4
    M = m.reshape(3,4)
    return M

M = resolverDLT3D(Q)
print(M)

#el error está en diagRH de A_monio, no pasa el test de simetria porque le sacamos el np.isclose que no esta permitido, 
# entonces devuelve None al chequear es Simetrica 
# opcion de solucion: # FORZAR SIMETRÍA NUMÉRICA
        #A_mon = (A_mon + traspuesta(A_mon)) / 2.0


[[ 6.44865265e-02  2.27816084e-02 -1.39906512e-02  8.37825121e-01]
 [ 3.69903837e-03  6.10180403e-02  1.31187332e-02  5.37810986e-01]
 [ 8.83537004e-05  4.47215926e-05  7.79036986e-05  5.11440314e-03]]


In [29]:
#c

## el determinante de R tiene que ser 1 (matriz de rotacion)? o esta bien que de -1 y sea una rotoreflexion
## pensamos poner pops como una lista con esos requisitos y leerla adentro de la funcion
def devolverMatricesCalibracion3d(M, pops=False):
    KR = M[:, :3]
    Kt = M[:, 3]
    Q, R_mono = calculaQR(inversa(KR))
    R = traspuesta(Q)
    K = inversa(R_mono)
    #elem = K[2,2]
    if pops:
        K[0,1] = 0
        prom = (K[0,0] + K[1,1])/2
        K[0,0] = prom
        K[1,1] = prom
    t = multiplicarAx(inversa(K), Kt)
    K[:, :] = K[:, :]/K[2,2]

    return R, t, K

'''
agregar elem en la funcion para chequear
R, t, K, elem = devolverMatricesCalibracion3d(M)
print(np.linalg.det(R))
print(R)
#print(np.linalg.inv(R))
print(matricesIguales(np.eye(3), multiplicarMatrices(R, R.T))) # R por R_tras es la identidad
print(matricesIguales(np.linalg.inv(R), R.T))
Rt = np.zeros((3,4))
Rt[:, :3] = R
Rt[:, 3] = t
print("verd")
print(matricesIguales(M, elem*K@Rt))
print(matricesIguales(M[:,:3], elem*K@R))
print(M[:,3])
print(K)
print(elem*K@t)'''
#da todo bien!!!!

'\nagregar elem en la funcion para chequear\nR, t, K, elem = devolverMatricesCalibracion3d(M)\nprint(np.linalg.det(R))\nprint(R)\n#print(np.linalg.inv(R))\nprint(matricesIguales(np.eye(3), multiplicarMatrices(R, R.T))) # R por R_tras es la identidad\nprint(matricesIguales(np.linalg.inv(R), R.T))\nRt = np.zeros((3,4))\nRt[:, :3] = R\nRt[:, 3] = t\nprint("verd")\nprint(matricesIguales(M, elem*K@Rt))\nprint(matricesIguales(M[:,:3], elem*K@R))\nprint(M[:,3])\nprint(K)\nprint(elem*K@t)'

### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


a) Obtener la matriz $\mathbf{K}$

b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.

 como W=0 para todo punto, ahora Q tiene 3 columnas que se anulan, Q esta en R^(2nx9)
 nuevamente queremos dim(Nu(Q)) = 1, por lo que rango(Q) = 9 - dim(Nu(Q)) = 8
 necesitamos 4 puntos, cada uno ocupa 2 columnas --> 8 filas LI de imagen, necesitamos que ninguna terna sea colineal (no existe una recta que contenga 3 puntos a la vez)

In [ ]:
carpeta = os.path.join("calib", "coplanar")
#coord_pix_fotos = np.loadtxt(os.path.join(carpeta, "detected_corners.txt"))
#coord_reales_puntos = np.loadtxt(os.path.join(carpeta, "p_W_corners.txt"), delimiter=",")

# 1 Q = 1 foto
# u1,v1,u2,v2... 
#a)
def obtenerQ2d(imgPts,objPts):
    #imgPts es un array que representa 1 sola foto
    #objPts es la matriz de coordenadas U,V,W de varios puntos en el mundo real
    n = len(objPts)
    Q = np.zeros((2*n,9))
    for i in range(n):
        #fila 1 del punto (coordenadas u)
        Q[2*i,:2] = objPts[i]
        Q[2*i,2] = 1
        u_i,v_i = imgPts[2*i], imgPts[2*i+1]
        Q[2*i,6:8] = -u_i*objPts[i]
        Q[2*i,8] = -u_i
        #fila 2 del mismo punto (coordenadas v)
        Q[2*i+1,3:6] = objPts[i]
        Q[2*i+1,6] = 1
        Q[2*i+1, 6:8] = -v_i*objPts[i]
        Q[2*i+1,8] = -v_i
    return Q


FileNotFoundError: calib/coplanar/detected_corners.txt not found.

In [ ]:
#armamos A con las homografias dadas
def armarV(x, y):
    v = np.array([x[0]*y[0],
                  x[0]*y[1] + y[0]*x[1],
                  x[1]*y[1],
                  x[0]*y[2] + y[0]*x[2],
                  x[1]*y[2] + y[1]*x[2],
                  x[2]*y[2]])
    return v

def armarA(Hlst):
    f = len(Hlst)
    A = np.zeros((2*f, 6))

    for i in range(f):
        h1 = Hlst[i][:,0]
        h2 = Hlst[i][:,1]
        a1 = armarV(h1,h2)
        a2 = armarV(h1,h1) - armarV(h2,h2)
        A[2*i] = a1
        A[2*i + 1] = a2
    return A

In [ ]:
#el minimo cuadrado homogeno es lo que hicimos antes
def calcularMatrizK2D(A):
    S, _ = diagRH(multiplicarMatrices(A.T, A))
    b = S[:, 5] #A es de 2nx6, AtA es de 6x6
    B = np.array([[b[0], b[1], b[3]],
              [b[1], b[2], b[4]],
              [b[3], b[4], b[5]]])

    #para que sea definida positiva la diagonal tiene que serlo
    if B[0][0] < 0:
        B = -B

    L = calculaCholesky(B)
    K = inversa(L.T)
    K = K/K[2,2]
    
    return K

In [ ]:
def productoCruz(x,y):
    v = np.array([x[1]*y[2] - x[2]*y[1],
                  x[2]*y[0] - x[0]*y[2],
                  x[0]*y[1] - x[1]*y[0]])
    return v

def obtenerPose2D(Hj, K):
    K_inv = inversa(K)
    K_invH = multiplicarMatrices(K_inv, Hj)
    columnasKiH = K_invH.T
    lamda = 1/norma(columnasKiH[0],2)
    
    r1 = lamda*columnasKiH[0]
    r2 = lamda*columnasKiH[1]

    tj = lamda*columnasKiH[2]

    #la imagen puede generarse detras de la camara pero tenemos que
    #generarla adelante, por eso le cambiamos el signo
    if tj[2] < 0:
        r1, r2, tj = -r1, -r2, -tj
    r3 = productoCruz(r1, r2)
    Rj = np.array([r1, r2, r3]).T

    return Rj, tj

### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---